# REPA × CIFAR-10 × MedDINOv3

Train a **SiT-S/8** diffusion transformer on CIFAR-10 using **MedDINOv3 ViT-Base** as the representation encoder (replacing DINOv2).

Steps:
1. Install dependencies
2. Clone REPA + MedDINOv3 repos
3. Download MedDINOv3 checkpoint
4. Pre-process CIFAR-10 (resize + VAE encode)
5. Launch training

## 1. Install dependencies

In [ ]:
%%bash
# Do NOT pin torch/torchvision — Kaggle already has the right CUDA build.
# Do NOT install xformers from the cu121 index — it downgrades torch.
pip install -q \
    accelerate \
    diffusers \
    timm \
    wandb \
    einops \
    huggingface_hub

## 2. Clone repos

In [ ]:
import os

WORK_DIR = '/kaggle/working'
REPA_DIR = os.path.join(WORK_DIR, 'REPA')
MEDDINOV3_DIR = os.path.join(WORK_DIR, 'MedDINOv3')
DATA_DIR = os.path.join(WORK_DIR, 'data', 'cifar10_256')
CKPT_DIR = os.path.join(WORK_DIR, 'meddinov3_ckpt')

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
%%bash -s "$WORK_DIR" "$REPA_DIR" "$MEDDINOV3_DIR"
WORK_DIR=$1
REPA_DIR=$2
MEDDINOV3_DIR=$3

cd $WORK_DIR

if [ ! -d "$REPA_DIR" ]; then
    git clone https://github.com/sihyun-yu/REPA.git REPA
fi

if [ ! -d "$MEDDINOV3_DIR" ]; then
    git clone https://github.com/comp-well-org/MedDINOv3.git MedDINOv3
fi

echo 'Repos ready.'

In [ ]:
# Copy the patched files from this notebook's directory into the cloned REPA repo.
# If you uploaded the patched files as a Kaggle dataset, adjust the src path below.

import shutil, os

PATCH_SRC = '/kaggle/input/repa-cifar10-patch'   # <-- upload patched files here as a dataset

patched_files = ['utils.py', 'train.py', 'dataset.py', 'prepare_cifar10.py']

if os.path.isdir(PATCH_SRC):
    for f in patched_files:
        src = os.path.join(PATCH_SRC, f)
        dst = os.path.join(REPA_DIR, f)
        if os.path.isfile(src):
            shutil.copy(src, dst)
            print(f'Copied {f}')
else:
    print('No patch dataset found — applying patches inline below.')

## 3. Download MedDINOv3 checkpoint

The checkpoint is available on [HuggingFace](https://huggingface.co/comp-well/MedDINOv3).  
Download `model.pth` and place it at `CKPT_DIR`.

In [ ]:
from huggingface_hub import hf_hub_download
import shutil, os

CKPT_PATH = os.path.join(CKPT_DIR, 'model.pth')

if not os.path.isfile(CKPT_PATH):
    local = hf_hub_download(
        repo_id='comp-well/MedDINOv3',
        filename='model.pth',
        local_dir=CKPT_DIR,
    )
    print(f'Downloaded to {local}')
else:
    print(f'Checkpoint already at {CKPT_PATH}')

## 4. Pre-process CIFAR-10 (resize to 256×256 + VAE encode)

In [ ]:
import sys
sys.path.insert(0, REPA_DIR)

os.environ['MEDDINOV3_PATH'] = MEDDINOV3_DIR
os.environ['MEDDINOV3_CKPT'] = CKPT_PATH

print('REPA dir:', REPA_DIR)
print('MedDINOv3 dir:', MEDDINOV3_DIR)
print('Checkpoint:', CKPT_PATH)

In [ ]:
%%bash -s "$REPA_DIR" "$DATA_DIR"
REPA_DIR=$1
DATA_DIR=$2

cd $REPA_DIR

python prepare_cifar10.py \
    --out-dir $DATA_DIR \
    --resolution 256 \
    --batch-size 128 \
    --cifar-root /tmp/cifar10_raw

echo 'Pre-processing done.'

In [ ]:
# Quick sanity check
import os, json, numpy as np

for split in ['train', 'val']:
    json_path = os.path.join(DATA_DIR, 'vae-sd', split, 'dataset.json')
    with open(json_path) as f:
        meta = json.load(f)['labels']
    sample_npy = os.path.join(DATA_DIR, 'vae-sd', split, meta[0][0])
    arr = np.load(sample_npy)
    print(f'{split}: {len(meta)} samples, latent shape: {arr.shape}')

## 5. Configure accelerate

In [ ]:
%%bash
# Single-GPU config for Kaggle (no DeepSpeed needed)
accelerate config default

## 6. Launch training

**Model**: `SiT-S/8` — smallest SiT variant, patch size 8  
**Encoder**: `meddinov3-vit-b` — MedDINOv3 ViT-Base (embed_dim=768)  
**Classes**: 10 (CIFAR-10)  
**Resolution**: 256 (CIFAR-10 images upscaled)  

In [ ]:
import subprocess, os

env = os.environ.copy()
env['MEDDINOV3_PATH'] = MEDDINOV3_DIR
env['MEDDINOV3_CKPT'] = CKPT_PATH

cmd = [
    'accelerate', 'launch',
    '--mixed_precision', 'fp16',
    '--num_processes', '1',
    os.path.join(REPA_DIR, 'train.py'),
    # experiment
    '--exp-name',            'repa-sits8-meddinov3-cifar10',
    '--output-dir',          os.path.join(WORK_DIR, 'exps'),
    '--report-to',           'wandb',
    # model
    '--model',               'SiT-S/8',
    '--num-classes',         '10',
    '--encoder-depth',       '8',
    '--enc-type',            'meddinov3-vit-b',
    # data
    '--data-dir',            DATA_DIR,
    '--resolution',          '256',
    '--batch-size',          '128',
    '--num-workers',         '2',
    # training schedule
    '--epochs',              '200',
    '--max-train-steps',     '100000',
    '--learning-rate',       '1e-4',
    '--mixed-precision',     'fp16',
    '--checkpointing-steps', '10000',
    '--sampling-steps',      '5000',
    # loss
    '--proj-coeff',          '0.5',
    '--cfg-prob',            '0.1',
    '--path-type',           'linear',
]

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, cwd=REPA_DIR, env=env)
print('Exit code:', result.returncode)

## 7. (Optional) Visualise generated samples

Checkpoints are saved to `exps/repa-sits8-meddinov3-cifar10/checkpoints/`.  
Use `generate.py` from the REPA repo to sample images.

In [ ]:
import glob, os

ckpt_dir = os.path.join(WORK_DIR, 'exps', 'repa-sits8-meddinov3-cifar10', 'checkpoints')
checkpoints = sorted(glob.glob(os.path.join(ckpt_dir, '*.pt')))
print('Available checkpoints:', checkpoints)

In [ ]:
# Generate samples from the latest checkpoint
if checkpoints:
    latest_ckpt = checkpoints[-1]
    result = subprocess.run([
        'python', os.path.join(REPA_DIR, 'generate.py'),
        '--model',       'SiT-S/8',
        '--num-classes', '10',
        '--ckpt',        latest_ckpt,
        '--cfg-scale',   '4.0',
        '--num-fid-samples', '1000',
        '--per-proc-batch-size', '64',
    ], cwd=REPA_DIR, env=env)
    print('Exit code:', result.returncode)